In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss
from sklearn.preprocessing import StandardScaler

# 📂 Define the data path
data_path = "data/"

# Load CSV files into DataFrames
conference_tourney_games = pd.read_csv(data_path + 'MConferenceTourneyGames.csv')
teams = pd.read_csv(data_path + 'MTeams.csv')
team_conferences = pd.read_csv(data_path + 'MTeamConferences.csv')
team_coaches = pd.read_csv(data_path + 'MTeamCoaches.csv')
secondary_tourney_teams = pd.read_csv(data_path + 'MSecondaryTourneyTeams.csv')
secondary_tourney_compact_results = pd.read_csv(data_path + 'MSecondaryTourneyCompactResults.csv')
seasons = pd.read_csv(data_path + 'MSeasons.csv')
regular_season_detailed_results = pd.read_csv(data_path + 'MRegularSeasonDetailedResults.csv')
regular_season_compact_results = pd.read_csv(data_path + 'MRegularSeasonCompactResults.csv')
ncaa_tourney_slots = pd.read_csv(data_path + 'MNCAATourneySlots.csv')
ncaa_tourney_seeds = pd.read_csv(data_path + 'MNCAATourneySeeds.csv')
ncaa_tourney_seed_round_slots = pd.read_csv(data_path + 'MNCAATourneySeedRoundSlots.csv')
ncaa_tourney_detailed_results = pd.read_csv(data_path + 'MNCAATourneyDetailedResults.csv')
ncaa_tourney_compact_results = pd.read_csv(data_path + 'MNCAATourneyCompactResults.csv')
game_cities = pd.read_csv(data_path + 'MGameCities.csv')

In [4]:
regular_season_detailed_results.columns

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='object')

In [9]:
regular_season_detailed_results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117748 entries, 0 to 117747
Data columns (total 34 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   Season   117748 non-null  int64 
 1   DayNum   117748 non-null  int64 
 2   WTeamID  117748 non-null  int64 
 3   WScore   117748 non-null  int64 
 4   LTeamID  117748 non-null  int64 
 5   LScore   117748 non-null  int64 
 6   WLoc     117748 non-null  object
 7   NumOT    117748 non-null  int64 
 8   WFGM     117748 non-null  int64 
 9   WFGA     117748 non-null  int64 
 10  WFGM3    117748 non-null  int64 
 11  WFGA3    117748 non-null  int64 
 12  WFTM     117748 non-null  int64 
 13  WFTA     117748 non-null  int64 
 14  WOR      117748 non-null  int64 
 15  WDR      117748 non-null  int64 
 16  WAst     117748 non-null  int64 
 17  WTO      117748 non-null  int64 
 18  WStl     117748 non-null  int64 
 19  WBlk     117748 non-null  int64 
 20  WPF      117748 non-null  int64 
 21  LFGM     1

In [8]:
regular_season_detailed_results.describe()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,NumOT,WFGM,WFGA,WFGM3,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
count,117748.000000,117748.000000,117748.00000,117748.000000,117748.000000,117748.000000,117748.000000,117748.000000,117748.000000,117748.000000,...,117748.000000,117748.000000,117748.000000,117748.000000,117748.000000,117748.000000,117748.000000,117748.000000,117748.000000,117748.000000
mean,2014.146355,70.294986,1288.25451,75.878936,1283.138830,63.888287,0.068689,26.401824,55.760242,7.347445,...,20.159790,12.073403,17.732454,10.461740,21.625650,11.409722,13.888907,6.004739,2.868185,19.305780
std,6.515929,35.772556,105.34750,10.998547,104.795432,10.848767,0.305098,4.680314,7.456374,3.119260,...,6.068136,5.344049,7.081056,4.221039,4.518197,3.724567,4.382700,2.745969,2.019050,4.553353
min,2003.000000,0.000000,1101.00000,34.000000,1101.000000,20.000000,0.000000,10.000000,26.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,4.000000,0.000000,0.000000,0.000000,0.000000,4.000000
25%,2009.000000,40.000000,1199.00000,68.000000,1192.000000,57.000000,0.000000,23.000000,51.000000,5.000000,...,16.000000,8.000000,13.000000,7.000000,19.000000,9.000000,11.000000,4.000000,1.000000,16.000000
50%,2014.000000,73.000000,1287.00000,75.000000,1282.000000,64.000000,0.000000,26.000000,55.000000,7.000000,...,20.000000,12.000000,17.000000,10.000000,21.000000,11.000000,14.000000,6.000000,3.000000,19.000000
75%,2020.000000,101.000000,1381.00000,83.000000,1374.000000,71.000000,0.000000,29.000000,60.000000,9.000000,...,24.000000,15.000000,22.000000,13.000000,25.000000,14.000000,17.000000,8.000000,4.000000,22.000000
max,2025.000000,132.000000,1480.00000,149.000000,1480.000000,144.000000,6.000000,57.000000,103.000000,26.000000,...,59.000000,48.000000,65.000000,36.000000,49.000000,31.000000,41.000000,22.000000,18.000000,45.000000


In [11]:
regular_season_compact_results.describe()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,NumOT
count,191796.000000,191796.000000,191796.000000,191796.000000,191796.000000,191796.000000,191796.000000
mean,2006.271257,73.602072,1287.599757,76.855664,1283.342286,64.771205,0.048937
std,11.623911,34.229230,104.920419,11.833224,105.102958,11.201711,0.258969
min,1985.000000,0.000000,1101.000000,34.000000,1101.000000,20.000000,0.000000
25%,1996.000000,45.000000,1199.000000,69.000000,1191.000000,57.000000,0.000000
50%,2007.000000,75.000000,1285.000000,76.000000,1281.000000,64.000000,0.000000
75%,2016.000000,103.000000,1380.000000,84.000000,1375.000000,72.000000,0.000000
max,2025.000000,132.000000,1480.000000,186.000000,1480.000000,150.000000,6.000000


In [14]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load datasets
tourney_results = pd.read_csv(data_path + "MNCAATourneyDetailedResults.csv")
regular_season_results = pd.read_csv(data_path + "MRegularSeasonDetailedResults.csv")

# Function to calculate efficiency stats for each game
def calculate_advanced_stats(df):
    df["WFG%"] = df["WFGM"] / df["WFGA"]
    df["LFG%"] = df["LFGM"] / df["LFGA"]
    df["W3P%"] = df["WFGM3"] / df["WFGA3"]
    df["L3P%"] = df["LFGM3"] / df["LFGA3"]
    df["WFT%"] = df["WFTM"] / df["WFTA"]
    df["LFT%"] = df["LFTM"] / df["LFTA"]

    df["WOR%"] = df["WOR"] / (df["WOR"] + df["LDR"])
    df["LOR%"] = df["LOR"] / (df["LOR"] + df["WDR"])
    df["WDR%"] = df["WDR"] / (df["WDR"] + df["LOR"])
    df["LDR%"] = df["LDR"] / (df["LDR"] + df["WOR"])

    df["WAST/TO"] = df["WAst"] / df["WTO"]
    df["LAST/TO"] = df["LAst"] / df["LTO"]

    df["WNetRating"] = 100 * df["WScore"] / (df["WFGA"] + 0.44 * df["WFTA"] + df["WTO"])
    df["LNetRating"] = 100 * df["LScore"] / (df["LFGA"] + 0.44 * df["LFTA"] + df["LTO"])

    return df

# Apply advanced stats function to both datasets
tourney_results = calculate_advanced_stats(tourney_results)
regular_season_results = calculate_advanced_stats(regular_season_results)

# Aggregate regular season performance per team per season
team_stats = regular_season_results.groupby(["Season", "WTeamID"]).agg(
    Wins=("WTeamID", "count"),
    AvgWinScore=("WScore", "mean"),
    AvgWinFGP=("WFG%", "mean"),
    AvgWin3PP=("W3P%", "mean"),
    AvgWinFT=("WFT%", "mean"),
    AvgWinORP=("WOR%", "mean"),
    AvgWinDRP=("WDR%", "mean"),
    AvgWinASTTO=("WAST/TO", "mean"),
    AvgWinNetRtg=("WNetRating", "mean"),
).reset_index()

# Rename columns for clarity
team_stats.rename(columns={"WTeamID": "TeamID"}, inplace=True)

# Merge regular season stats with tournament results
tourney_results = tourney_results.merge(team_stats, left_on=["Season", "WTeamID"], right_on=["Season", "TeamID"], how="left")
tourney_results = tourney_results.merge(team_stats, left_on=["Season", "LTeamID"], right_on=["Season", "TeamID"], how="left", suffixes=("_W", "_L"))

# Calculate performance differences
tourney_results["Diff_Score"] = tourney_results["AvgWinScore_W"] - tourney_results["AvgWinScore_L"]
tourney_results["Diff_FGP"] = tourney_results["AvgWinFGP_W"] - tourney_results["AvgWinFGP_L"]
tourney_results["Diff_3PP"] = tourney_results["AvgWin3PP_W"] - tourney_results["AvgWin3PP_L"]
tourney_results["Diff_FT"] = tourney_results["AvgWinFT_W"] - tourney_results["AvgWinFT_L"]
tourney_results["Diff_ORP"] = tourney_results["AvgWinORP_W"] - tourney_results["AvgWinORP_L"]
tourney_results["Diff_DRP"] = tourney_results["AvgWinDRP_W"] - tourney_results["AvgWinDRP_L"]
tourney_results["Diff_ASTTO"] = tourney_results["AvgWinASTTO_W"] - tourney_results["AvgWinASTTO_L"]
tourney_results["Diff_NetRtg"] = tourney_results["AvgWinNetRtg_W"] - tourney_results["AvgWinNetRtg_L"]

# Save processed dataset for modeling
#tourney_results.to_csv("Processed_Tourney_Stats.csv", index=False)
tourney_results


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,AvgWinASTTO_L,AvgWinNetRtg_L,Diff_Score,Diff_FGP,Diff_3PP,Diff_FT,Diff_ORP,Diff_DRP,Diff_ASTTO,Diff_NetRtg
0,2003,134,1421,92,1411,84,N,1,32,69,...,1.004138,92.110577,2.205128,-0.002992,0.057066,0.131790,0.001807,-0.003006,0.174482,3.852371
1,2003,136,1112,80,1436,51,N,0,31,66,...,1.225627,91.936500,14.410526,0.001655,-0.012322,0.033150,-0.006888,-0.073551,0.119045,3.195739
2,2003,136,1113,84,1272,71,N,0,31,59,...,1.367832,90.066599,5.896135,0.073229,-0.021237,0.069926,0.052960,0.004546,-0.078327,8.362731
3,2003,136,1141,79,1166,73,N,0,29,53,...,1.405176,100.869760,2.658171,0.024323,0.012983,0.091808,0.030467,-0.028198,-0.461274,0.461835
4,2003,136,1143,76,1301,74,N,1,27,64,...,1.249668,102.121235,-1.793651,-0.001011,-0.015598,-0.080826,0.000242,0.019862,0.001474,-7.421269
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377,2024,146,1301,76,1181,64,N,0,28,60,...,1.980101,105.616132,-0.651515,-0.021504,-0.054352,0.014659,-0.056506,-0.016386,-0.243901,-3.287960
1378,2024,146,1345,72,1397,66,N,0,24,53,...,2.033998,102.108083,1.622126,0.029904,0.049108,-0.056252,0.030002,0.017125,-0.107633,3.222092
1379,2024,152,1163,86,1104,72,N,0,31,62,...,1.735119,111.128269,-12.425499,0.000012,-0.037535,-0.049115,-0.010873,0.046317,0.607348,-2.293269
1380,2024,152,1345,63,1301,50,N,0,22,55,...,1.736200,102.328172,3.731975,0.022063,0.071393,-0.020275,0.104465,0.014490,0.190166,3.002004


In [36]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import StandardScaler

# 📂 Define the data path
data_path = "data/"  # Ensure your CSV files are in this folder

### === FUNCTION TO PROCESS TOURNAMENT DATA === ###
def process_avg_teamstats_data(tourney_filename, season_filename):
    """
    Process tournament and regular season data for either men's or women's NCAA tournament.
    """
    # Load datasets
    tourney_results = pd.read_csv(os.path.join(data_path, tourney_filename))
    season_results = pd.read_csv(os.path.join(data_path, season_filename))
    
    # Function to calculate efficiency stats
    def calculate_advanced_stats(df):
        df["FG%"] = (df["WFGM"] + df["LFGM"]) / (df["WFGA"] + df["LFGA"])
        df["3P%"] = (df["WFGM3"] + df["LFGM3"]) / (df["WFGA3"] + df["LFGA3"])
        df["FT%"] = (df["WFTM"] + df["LFTM"]) / (df["WFTA"] + df["LFTA"])
        df["OR%"] = (df["WOR"] + df["LOR"]) / (df["WOR"] + df["LOR"] + df["WDR"] + df["LDR"])
        df["DR%"] = (df["WDR"] + df["LDR"]) / (df["WOR"] + df["LOR"] + df["WDR"] + df["LDR"])
        df["AST/TO"] = (df["WAst"] + df["LAst"]) / (df["WTO"] + df["LTO"])
        df["NetRating"] = 100 * (df["WScore"] + df["LScore"]) / (df["WFGA"] + df["LFGA"] + 0.44 * (df["WFTA"] + df["LFTA"]) + df["WTO"] + df["LTO"])
        return df

    # Apply advanced stats
    tourney_results = calculate_advanced_stats(tourney_results)
    season_results = calculate_advanced_stats(season_results)

    # Aggregate season performance
    team_stats = season_results.groupby(["Season", "WTeamID"]).agg(
        Wins=("WTeamID", "count"),
        AvgScore=("WScore", "mean"),
        AvgFGP=("FG%", "mean"),
        Avg3PP=("3P%", "mean"),
        AvgFT=("FT%", "mean"),
        AvgORP=("OR%", "mean"),
        AvgDRP=("DR%", "mean"),
        AvgASTTO=("AST/TO", "mean"),
        AvgNetRtg=("NetRating", "mean"),
    ).reset_index()
    
    # Rename column
    team_stats.rename(columns={"WTeamID": "TeamID"}, inplace=True)
    
    return team_stats

### === PROCESS MEN'S AND WOMEN'S DATA === ###
men_teamstats = process_avg_teamstats_data("MNCAATourneyDetailedResults.csv", "MRegularSeasonDetailedResults.csv")
women_teamstats = process_avg_teamstats_data("WNCAATourneyDetailedResults.csv", "WRegularSeasonDetailedResults.csv")




In [23]:
import pandas as pd

# Load the CSV files into DataFrames
team_stats = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/Processed_Men_TeamStats.csv')
print(team_stats.head())

regular_season_results = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/MRegularSeasonCompactResults.csv')
print(regular_season_results.head())

tourney_results = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/MNCAATourneyCompactResults.csv')
print(tourney_results.head())

# Merge regular season and tournament results first
merged_results = pd.concat([regular_season_results, tourney_results])

# Merge team stats with the combined results for winning teams
merged_results = merged_results.merge(team_stats, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'], suffixes=('', '_W'))
merged_results = merged_results.merge(team_stats, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'], suffixes=('', '_L'))

# Drop the redundant 'TeamID' columns
merged_results.drop(columns=['TeamID', 'TeamID_L'], inplace=True)

# save as csv
merged_results.to_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/Merged_Results.csv', index=False)

# Load the CSV files into DataFrames for women's data
women_team_stats = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/Processed_Women_TeamStats.csv')
print(women_team_stats.head())

women_regular_season_results = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/WRegularSeasonCompactResults.csv')
print(women_regular_season_results.head())

women_tourney_results = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/WNCAATourneyCompactResults.csv')
print(women_tourney_results.head())

# Merge regular season and tournament results first for women's data
women_merged_results = pd.concat([women_regular_season_results, women_tourney_results])

# Merge team stats with the combined results for winning teams
women_merged_results = women_merged_results.merge(women_team_stats, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'], suffixes=('', '_W'))
women_merged_results = women_merged_results.merge(women_team_stats, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'], suffixes=('', '_L'))

# Drop the redundant 'TeamID' columns
women_merged_results.drop(columns=['TeamID', 'TeamID_L'], inplace=True)
# Merge men and women merged_results
final_merged = pd.concat([merged_results, women_merged_results])
print(final_merged.shape)


   Season  TeamID  Wins   AvgScore    AvgFGP    Avg3PP     AvgFT    AvgORP  \
0    2003    1102    12  68.750000  0.480108  0.408717  0.666554  0.282500   
1    2003    1103    13  87.769231  0.499673  0.334016  0.740982  0.352170   
2    2003    1104    17  74.705882  0.414552  0.322790  0.711241  0.331314   
3    2003    1105     7  79.428571  0.405907  0.332620  0.713975  0.359001   
4    2003    1106    13  68.307692  0.401399  0.333188  0.658466  0.334422   

     AvgDRP  AvgASTTO  AvgNetRtg  
0  0.717500  1.037241  95.662871  
1  0.647830  1.137838  99.123317  
2  0.668686  0.915814  84.922891  
3  0.640999  0.732756  80.583425  
4  0.665578  0.685807  78.286200  
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1985      20     1228      81     1328      64    N      0
1    1985      25     1106      77     1354      70    H      0
2    1985      25     1112      63     1223      56    H      0
3    1985      25     1165      70     1432      54    H      0
4

In [30]:
final_merged.to_csv('final_merged.csv', index=False)

In [21]:
df = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/Merged_Results.csv')

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split
df = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/Merged_Results.csv')
# Load and prepare data (assuming 'df' is your DataFrame)
X = df.drop(['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT'], axis=1)
y = (df['WTeamID'] < df['LTeamID']).astype(int)  # Win probability for lower TeamId

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict_proba(X_val_scaled)[:, 1]
print("Logistic Regression - Brier Score:", brier_score_loss(y_val, y_pred_lr))

# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict_proba(X_val_scaled)[:, 1]
print("Random Forest - Brier Score:", brier_score_loss(y_val, y_pred_rf))

Logistic Regression - Brier Score: 0.2488404518170671
Random Forest - Brier Score: 0.1421043814649735


In [35]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer, brier_score_loss
import warnings
warnings.filterwarnings("ignore")

# Load data
df = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/final_merged.csv')

# Prepare data
X = df.drop(['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT'], axis=1)
y = (df['WTeamID'] < df['LTeamID']).astype(int)  # Win probability for lower TeamId

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict_proba(X_val_scaled)[:, 1]
print("Logistic Regression:")
print("  - Brier Score:", brier_score_loss(y_val, y_pred_lr))
print("  - Accuracy:", accuracy_score(y_val, (y_pred_lr >= 0.5).astype(int)))

# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict_proba(X_val_scaled)[:, 1]
print("Random Forest (initial):")
print("  - Brier Score:", brier_score_loss(y_val, y_pred_rf))
print("  - Accuracy:", accuracy_score(y_val, (y_pred_rf >= 0.5).astype(int)))

# Hyperparameter tuning for Random Forest using RandomizedSearchCV
param_distributions = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5, 10]
}

brier_scorer = make_scorer(brier_score_loss, greater_is_better=False, needs_proba=True)

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), 
    param_distributions, 
    cv=5, 
    scoring=brier_scorer, 
    n_iter=10, 
    n_jobs=-1  # Enable parallel processing
)
random_search.fit(X_train_scaled, y_train)

print("Random Forest (tuned):")
print("  - Best Parameters:", random_search.best_params_)
print("  - Best Brier Score:", random_search.best_score_)
print("  - Accuracy (best params):", accuracy_score(y_val, (random_search.best_estimator_.predict_proba(X_val_scaled)[:, 1] >= 0.5).astype(int)))

Logistic Regression:
  - Brier Score: 0.24954586880533097
  - Accuracy: 0.5167657788216481
Random Forest (initial):
  - Brier Score: 0.14214639948028585
  - Accuracy: 0.8188246464444555
Random Forest (tuned):
  - Best Parameters: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 10, 'max_depth': None}
  - Best Brier Score: nan
  - Accuracy (best params): 0.8074059267402929


In [37]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer, brier_score_loss
import warnings
import itertools
import numpy as np

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer, brier_score_loss
import warnings
import itertools
import numpy as np

warnings.filterwarnings("ignore")

# Load data
df = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/final_merged.csv')

# Prepare data
X = df.drop(['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT'], axis=1)
y = (df['WTeamID'] < df['LTeamID']).astype(int)  # Win probability for lower TeamId

# Split data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict_proba(X_val_scaled)[:, 1]
print("Logistic Regression:")
print("  - Brier Score:", brier_score_loss(y_val, y_pred_lr))
print("  - Accuracy:", accuracy_score(y_val, (y_pred_lr >= 0.5).astype(int)))

# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict_proba(X_val_scaled)[:, 1]
print("Random Forest (initial):")
print("  - Brier Score:", brier_score_loss(y_val, y_pred_rf))
print("  - Accuracy:", accuracy_score(y_val, (y_pred_rf >= 0.5).astype(int)))

# Hyperparameter tuning for Random Forest using RandomizedSearchCV
param_distributions = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5, 10]
}

brier_scorer = make_scorer(brier_score_loss, greater_is_better=False, needs_proba=True)

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), 
    param_distributions, 
    cv=5, 
    scoring=brier_scorer, 
    n_iter=10, 
    n_jobs=-1  # Enable parallel processing
)
random_search.fit(X_train_scaled, y_train)

print("Random Forest (tuned):")
print("  - Best Parameters:", random_search.best_params_)
print("  - Best Brier Score:", random_search.best_score_)
print("  - Accuracy (best params):", accuracy_score(y_val, (random_search.best_estimator_.predict_proba(X_val_scaled)[:, 1] >= 0.5).astype(int)))



Logistic Regression:
  - Brier Score: 0.24954586880533097
  - Accuracy: 0.5167657788216481
Random Forest (initial):
  - Brier Score: 0.14214639948028585
  - Accuracy: 0.8188246464444555
Random Forest (tuned):
  - Best Parameters: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_depth': 10}
  - Best Brier Score: nan
  - Accuracy (best params): 0.6330268352406176


TypeError: Could not convert ['NHHNHHNHHHHHHHAHHHHHHHNHHHHAHAAHANHHNHAAHHHHHAHAAHAHHHHANHHHHHHHAAHHHHAHHHHNNNHHHHNAHHHHHAHHHHHAHHHHHNNHHHHHHHNHNHHHHHHHHHHHAANHHNNHHHHHAAHHHAHNHHHHHHHHAHHAHHAHAHHHNHHNNNHHHNHAAHHHAHAHHNHNNHHAHHAHHHAHAHHHHNHHHHHHHHHHHHHHHNHHHHHHAHHAAHAHHNNAAHHHHHAHHAAHNHHNHHNHAHAHAHAAHHNNNHHHNHHNHHAHHHAHHNNHHNNHHNHAHAHHHAHANHHNHANHHHHAHAHAHHNNHHHAHAAHAHHHAHHAHANNNHHHHNNAHHHAHHHAHHAHHHAHNNHAHHAHHAHAAHHAHAHAHHNNNHHHHNHHNAHAHHHAHAHHAHHHHNNNAHAHHHAAAHHAHAANNNNNNNNNNNNN'] to numeric

In [49]:
# Generate submission file
team_ids = df['WTeamID'].unique()  # Assuming WTeamID has all unique team IDs
matchups = list(itertools.combinations(team_ids, 2))

submission_predictions = []
for matchup in matchups:
    # Ensure the lower TeamId is first
    if matchup[0] > matchup[1]:
        matchup = (matchup[1], matchup[0])
    
    # Create the matchup ID
    matchup_id = f"2025_{matchup[0]}_{matchup[1]}"
    
    # Predict the probability of the lower TeamId winning
    # Assuming the features for the matchup can be generated using the following function
    input_features = generate_input_features(matchup[0], matchup[1], df, scaler)
    prediction = random_search.best_estimator_.predict_proba(input_features)[:, 1][0]
    
    # Append the predicted probability to the list
    submission_predictions.append((matchup_id, prediction))

# Create the submission DataFrame
submission_df = pd.DataFrame(submission_predictions, columns=["ID", "Pred"])

# Save the submission file to a CSV
submission_df.to_csv("submission.csv", index=False)


def generate_input_features(team_id_1, team_id_2, df, scaler):
    # Select only numeric columns
    numeric_df = df.select_dtypes(include=['int64', 'float64'])
    
    # Drop 'WTeamID' column since we're filtering by it
    numeric_df = numeric_df.drop('WTeamID', axis=1)
    
    # Calculate the mean for each team
    team_1_features = numeric_df[df['WTeamID'] == team_id_1].mean().values
    team_2_features = numeric_df[df['WTeamID'] == team_id_2].mean().values
    
    # Check if team_1_features or team_2_features is empty
    if len(team_1_features) == 0 or len(team_2_features) == 0:
        # If either team has no data, return a default feature set
        return scaler.transform([[0.0] * (len(numeric_df.columns) * 2)])
    
    # Concatenate the features
    features = np.concatenate((team_1_features, team_2_features))
    
    # Scale the features
    return scaler.transform([features])

TypeError: Could not convert ['NHHNHHNHHHHHHHAHHHHHHHNHHHHAHAAHANHHNHAAHHHHHAHAAHAHHHHANHHHHHHHAAHHHHAHHHHNNNHHHHNAHHHHHAHHHHHAHHHHHNNHHHHHHHNHNHHHHHHHHHHHAANHHNNHHHHHAAHHHAHNHHHHHHHHAHHAHHAHAHHHNHHNNNHHHNHAAHHHAHAHHNHNNHHAHHAHHHAHAHHHHNHHHHHHHHHHHHHHHNHHHHHHAHHAAHAHHNNAAHHHHHAHHAAHNHHNHHNHAHAHAHAAHHNNNHHHNHHNHHAHHHAHHNNHHNNHHNHAHAHHHAHANHHNHANHHHHAHAHAHHNNHHHAHAAHAHHHAHHAHANNNHHHHNNAHHHAHHHAHHAHHHAHNNHAHHAHHAHAAHHAHAHAHHNNNHHHHNHHNAHAHHHAHAHHAHHHHNNNAHAHHHAAAHHAHAANNNNNNNNNNNNN'] to numeric

In [45]:
df['WTeamID'].unique()

array([1104, 1272, 1266, 1296, 1400, 1458, 1161, 1186, 1194, 1166, 1202,
       1237, 1323, 1125, 1156, 1183, 1314, 1353, 1390, 1426, 1462, 1196,
       1242, 1422, 1304, 1113, 1116, 1117, 1120, 1122, 1123, 1133, 1139,
       1140, 1141, 1148, 1150, 1151, 1157, 1158, 1160, 1165, 1179, 1187,
       1191, 1200, 1207, 1211, 1217, 1218, 1219, 1232, 1241, 1247, 1250,
       1258, 1261, 1265, 1267, 1277, 1281, 1298, 1301, 1308, 1317, 1320,
       1321, 1328, 1329, 1340, 1343, 1345, 1348, 1349, 1350, 1352, 1365,
       1373, 1375, 1379, 1382, 1384, 1394, 1395, 1397, 1403, 1404, 1412,
       1425, 1427, 1428, 1435, 1436, 1437, 1438, 1451, 1452, 1112, 1124,
       1127, 1131, 1143, 1145, 1153, 1163, 1177, 1178, 1181, 1185, 1198,
       1208, 1209, 1210, 1225, 1226, 1233, 1240, 1257, 1259, 1273, 1274,
       1319, 1335, 1338, 1341, 1344, 1351, 1360, 1371, 1386, 1392, 1409,
       1418, 1420, 1429, 1440, 1442, 1444, 1450, 1453, 1455, 1460, 1461,
       1114, 1138, 1155, 1169, 1173, 1199, 1204, 12

## OPENAI

In [50]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer
import warnings
import numpy as np

warnings.filterwarnings("ignore")

# Load data
df = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/final_merged.csv')

# Prepare data
X = df.drop(['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT'], axis=1)
y = (df['WTeamID'] < df['LTeamID']).astype(int)  # Win probability for lower TeamId

# Split data (stratify ensures balanced classes)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature scaling (only for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict_proba(X_val_scaled)[:, 1]

print("Logistic Regression:")
print("  - Brier Score:", brier_score_loss(y_val, y_pred_lr))
print("  - Accuracy:", accuracy_score(y_val, (y_pred_lr >= 0.5).astype(int)))

# Random Forest Classifier (without scaling)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict_proba(X_val)[:, 1]

print("Random Forest (initial):")
print("  - Brier Score:", brier_score_loss(y_val, y_pred_rf))
print("  - Accuracy:", accuracy_score(y_val, (y_pred_rf >= 0.5).astype(int)))

# Hyperparameter tuning for Random Forest using RandomizedSearchCV
param_distributions = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5, 10]
}

brier_scorer = make_scorer(brier_score_loss, greater_is_better=False, needs_proba=True)

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions,
    cv=5,
    scoring=brier_scorer,
    n_iter=10,
    n_jobs=-1  # Parallel processing
)
random_search.fit(X_train, y_train)

# Predict using the best Random Forest model
y_pred_rf_tuned = random_search.best_estimator_.predict_proba(X_val)[:, 1]

# Check for NaN values in predictions
if np.isnan(y_pred_rf_tuned).any():
    print("Warning: NaN values found in probability predictions!")

print("Random Forest (tuned):")
print("  - Best Parameters:", random_search.best_params_)
print("  - Accuracy (best params):", accuracy_score(y_val, (y_pred_rf_tuned >= 0.5).astype(int)))
print("  - Brier Score:", brier_score_loss(y_val, y_pred_rf_tuned))


Logistic Regression:
  - Brier Score: 0.24940398217546683
  - Accuracy: 0.5232122332717005
Random Forest (initial):
  - Brier Score: 0.1437074958772675
  - Accuracy: 0.8144520513717456
Random Forest (tuned):
  - Best Parameters: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_depth': None}
  - Accuracy (best params): 0.8226725301084403
  - Brier Score: 0.16461167446176553


In [78]:
y_val

150587    1
85022     0
109087    0
70385     0
176785    1
         ..
114698    1
3454      1
175730    1
88721     1
137032    0
Length: 40022, dtype: int32

In [77]:
y_pred_rf_tuned

array([0.56615277, 0.45536653, 0.54615881, ..., 0.44945639, 0.71167611,
       0.23056117])

In [ ]:

# Load submission file
submission_df = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/SampleSubmissionStage2.csv')

# Extract Season, LowerID, HigherID from 'ID'
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True)
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df[['Season', 'LowerID', 'HigherID']].astype(int)

# Load dataset
df = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/final_merged.csv')

# Merge based on LowerID and HigherID
test_matches = submission_df.merge(df, how='left', left_on=['Season', 'LowerID', 'HigherID'], right_on=['Season', 'WTeamID', 'LTeamID'])

# Merge the alternative order (HigherID, LowerID)
test_matches_alt = submission_df.merge(df, how='left', left_on=['Season', 'LowerID', 'HigherID'], right_on=['Season', 'LTeamID', 'WTeamID'])

# Combine both merge results
test_matches = test_matches.combine_first(test_matches_alt)

# ✅ Drop duplicates
test_matches = test_matches.drop_duplicates(subset=['ID'])

# ✅ Ensure only valid matchups are kept
test_matches = test_matches[test_matches['ID'].isin(submission_df['ID'])]

# Drop non-numeric columns
X_test = test_matches.drop(['ID', 'Season', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT', 'LowerID', 'HigherID'], axis=1, errors='ignore')

# ✅ Ensure all remaining columns are numeric
X_test = X_test.select_dtypes(include=[np.number])

# ✅ Fix Feature Mismatch: Keep only trained features
trained_features = random_search.best_estimator_.feature_names_in_  # Get trained feature names
X_test = X_test[trained_features]  # Select only those columns

# Check if X_test is empty after filtering
if X_test.empty:
    raise ValueError("Error: X_test is empty after removing non-matching features.")

# **Load trained scaler (use .transform() if already fitted)**
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test)  

# **Use the best-tuned Random Forest model to predict win probabilities**
y_test_pred = random_search.best_estimator_.predict_proba(X_test_scaled)[:, 1]

# **Ensure predictions match submission file length**
if len(y_test_pred) != len(submission_df):
    print(f"Warning: Mismatch in lengths. Predictions: {len(y_test_pred)}, Submission: {len(submission_df)}")
    y_test_pred = y_test_pred[:len(submission_df)]

# Assign predictions
submission_df['Pred'] = y_test_pred

# Save final submission CSV
submission_df[['ID', 'Pred']].to_csv('final_submission.csv', index=False)

print("✅ Submission file saved as 'final_submission.csv'.")


✅ Submission file saved as 'final_submission.csv'.


In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ------------------- Data Loading & Preprocessing --------------------

# Load data
df = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/final_merged.csv')

# Prepare the features and target
drop_cols = ['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT']
X = df.drop(columns=drop_cols, errors='ignore')
y = (df['WTeamID'] < df['LTeamID']).astype(int)  # Lower TeamID wins

# Stratified train-test split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ------------------- Model Training & Hyperparameter Tuning --------------------

# Initial Random Forest Classifier without scaling
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=5, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate baseline
y_pred_rf = rf_model.predict_proba(X_val)[:, 1]
print("Random Forest (initial):")
print(" - Brier Score:", brier_score_loss(y_val, y_pred_rf))
print(" - Accuracy:", accuracy_score(y_val, (y_pred_rf >= 0.5).astype(int)))

# Hyperparameter tuning using Brier Score explicitly
param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5, 10]
}

brier_scorer = make_scorer(brier_score_loss, greater_is_better=False, needs_proba=True)

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions,
    cv=5,
    scoring=brier_scorer,
    n_iter=10,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

print("\nRandom Forest (tuned):")
print(" - Best Parameters:", random_search.best_params_)

y_pred_rf_tuned = random_search.best_estimator_.predict_proba(X_val)[:, 1]
print(" - Brier Score:", brier_score_loss(y_val, y_pred_rf_tuned))
print(" - Accuracy:", accuracy_score(y_val, (y_pred_rf_tuned >= 0.5).astype(int)))

# ------------------- Generate Predictions for Submission --------------------

# Load submission template
submission_df = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/SampleSubmissionStage2.csv')
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True).astype(int)

# Merge matchup data
test_matches = submission_df.merge(df, how='left', left_on=['Season', 'LowerID', 'HigherID'], right_on=['Season', 'WTeamID', 'LTeamID'])
test_matches_alt = submission_df.merge(df, how='left', left_on=['Season', 'LowerID', 'HigherID'], right_on=['Season', 'LTeamID', 'WTeamID'])
test_matches = test_matches.combine_first(test_matches_alt)

# Remove duplicates and keep only matched rows
test_matches = test_matches.drop_duplicates(subset=['ID'])
test_matches = test_matches[test_matches['ID'].isin(submission_df['ID'])]

# Prepare test features explicitly aligned with training
trained_features = random_search.best_estimator_.feature_names_in_
X_test = test_matches[trained_features]

# Handle missing data (if any)
if X_test.isnull().sum().sum() > 0:
    X_test = X_test.fillna(X_train.mean())

# Predict probabilities
y_test_pred = random_search.best_estimator_.predict_proba(X_test)[:, 1]

# Verify prediction distribution
print("Predictions Stats (Test Data):")
print(" - Min:", y_test_pred.min(), "Max:", y_test_pred.max(), "Mean:", y_test_pred.mean())

# Assign predictions to submission
submission_df['Pred'] = y_test_pred

# Save final submission
submission_df[['ID', 'Pred']].to_csv('final_submission.csv', index=False)
print("✅ Submission file saved as 'final_submission.csv'.")


Random Forest (initial):
 - Brier Score: 0.2472837591911843
 - Accuracy: 0.5478486832242266

Random Forest (tuned):
 - Best Parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_depth': None}
 - Brier Score: 0.164211780093617
 - Accuracy: 0.8251461696067163
Predictions Stats (Test Data):
 - Min: 0.09271765856262766 Max: 0.9371649933399927 Mean: 0.4391005314392999
✅ Submission file saved as 'final_submission.csv'.


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# Load training data
train_df = pd.read_csv('final_merged.csv')

# Prepare training features and labels
# Define stat columns (no scaling needed for tree-based models)
stat_cols = ['Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT', 
             'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg']
training_rows = []

# For each game, set team1 as the team with the smaller ID for consistency
# Outcome = 1 if team1 wins, 0 if team1 loses
for _, row in train_df.iterrows():
    season = row['Season']
    w_id = row['WTeamID']
    l_id = row['LTeamID']
    if w_id < l_id:
        team1_id, team2_id = w_id, l_id
        outcome = 1  # team1 (lower ID) won
        team1_stats = [row[col]        for col in stat_cols]      # winner's stats
        team2_stats = [row[col + '_L'] for col in stat_cols]      # loser's stats
    else:
        team1_id, team2_id = l_id, w_id
        outcome = 0  # team1 (lower ID) lost
        team1_stats = [row[col + '_L'] for col in stat_cols]      # loser's stats
        team2_stats = [row[col]        for col in stat_cols]      # winner's stats
    training_rows.append([season, team1_id, team2_id] + team1_stats + team2_stats + [outcome])

# Create DataFrame for training data
team1_cols = [col + '_1' for col in stat_cols]
team2_cols = [col + '_2' for col in stat_cols]
columns    = ['Season', 'Team1', 'Team2'] + team1_cols + team2_cols + ['Outcome']
train_data = pd.DataFrame(training_rows, columns=columns)

# Separate features and target label
X = train_data[team1_cols + team2_cols]
y = train_data['Outcome']

# Initialize RandomForest with limited complexity to reduce overfitting
# (StandardScaler removed as it's unnecessary for tree models)
rf_model = RandomForestClassifier(max_depth=5, min_samples_leaf=5, random_state=42)

# Set up hyperparameter search for n_estimators, using log loss for scoring
param_dist = {
    'n_estimators': [100, 200, 500, 1000]  # number of trees to try
    # max_depth and min_samples_leaf are fixed to the values above
}
search_cv = RandomizedSearchCV(rf_model, param_distributions=param_dist, 
                               n_iter=4, cv=5, scoring='neg_log_loss', 
                               random_state=42, n_jobs=-1)
search_cv.fit(X, y)

# Get the best model
best_rf = search_cv.best_estimator_



FileNotFoundError: [Errno 2] No such file or directory: '/data/SampleSubmissionStage2.csv'

In [4]:
# Load sample submission and prepare for predictions
submission_df = pd.read_csv('c:/Users/JonMa/OneDrive/portfolio/Madness/data/SampleSubmissionStage2.csv')
predictions = []

# Create a lookup dictionary for team stats by (Season, TeamID)
team_stats = {}
for _, row in train_df.iterrows():
    season = row['Season']
    w_id, l_id = row['WTeamID'], row['LTeamID']
    team_stats[(season, w_id)] = [row[col]        for col in stat_cols]      # winner stats
    team_stats[(season, l_id)] = [row[col + '_L'] for col in stat_cols]      # loser stats

# Predict probabilities for each matchup in the submission file
for id_str in submission_df['ID']:
    # Parse the matchup ID "Season_Team1_Team2"
    season, team1_id, team2_id = id_str.split('_')
    season   = int(season)
    team1_id = int(team1_id)
    team2_id = int(team2_id)
    # Ensure team1_id is the lower ID (should be already, given ID format)
    if team1_id > team2_id:
        team1_id, team2_id = team2_id, team1_id
    # Retrieve team stats (if a team is missing, use zeros or compute as needed)
    stats_team1 = team_stats.get((season, team1_id), [0]*len(stat_cols))
    stats_team2 = team_stats.get((season, team2_id), [0]*len(stat_cols))
    # Form feature vector for the matchup and predict probability of team1 winning
    match_features = stats_team1 + stats_team2
    prob_team1_wins = best_rf.predict_proba([match_features])[0][1]
    predictions.append(prob_team1_wins)

# Attach predictions and optionally examine their range
submission_df['Pred'] = predictions
print(f"Predicted probability range: {min(predictions):.3f} to {max(predictions):.3f}")
print(submission_df.head(10))  # sample output

# Save the predictions to a CSV file (for actual submission)
# submission_df.to_csv('MySubmission.csv', index=False)


Predicted probability range: 0.109 to 0.890
               ID      Pred
0  2025_1101_1102  0.539666
1  2025_1101_1103  0.269739
2  2025_1101_1104  0.246416
3  2025_1101_1105  0.527865
4  2025_1101_1106  0.454647
5  2025_1101_1107  0.455680
6  2025_1101_1108  0.516089
7  2025_1101_1110  0.297189
8  2025_1101_1111  0.387625
9  2025_1101_1112  0.273081
